#### Middleware

Middleware provides a method to understand in depth about what happens inside the agent. It is useful for the following: 
- Tracking agent behaviour with logging, analytics, debugging
- Transforming prompts, tool selection, and output formatting
- Adding retries, fallbacks, and early termination logic
- Applying rate limits, guardrails, and Pll detection

Middleware acts like checks/triggers.

Built in middlewares:
- **Summarization** - based on input provided and output generated, it returns the summary of operations performed until the condition is true.
- **Human in the (loop) feedback** 
- **Model Call Limit** 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [ ]:
### sumarization middleware

from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

### message based summarization

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=("messages",10),
        keep=("messages",4)
    )]
)

In [15]:
### run with thread id
config = {'configurable':{'thread_id':'test_1'}}

In [ ]:
questions = [
    'What is 2+2?',
    'What is 2-2?',
    'What is 2*2?',
    'What is 2/2?',
    'What is 22+2?',
    'What is 77-2?',
    'What is 12+2?',
    'What is 2+24?',
    'What is 24/2?',
]

for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )

    print("Question:", q)
    print("Answer:", response["messages"][-1].content)
    print("-" * 50)

#### Token Size

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_ollama import ChatOllama


@tool
def search_hotels(city:str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f"""Hotels in {city}"""

model = ChatOllama(model='qwen2.5:3b')

agent3 = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=("tokens",550),
        keep=("tokens",200)
    )]
)

In [24]:
config = {'configurable':{'thread_id':'test_1'}}

def token_count(messages):
    total = sum(len(str(m.content)) for m in messages)
    return total // 4



cities = ['Pulbangash', 'Karkarduma', 'Pitampura', 'Rithala', 'Noida', 'Dwarka Mor', 'Uttam Nagar']

for city in cities:
    response = agent3.invoke(
        {"messages": [HumanMessage(content=f'Find hotels in {city}')]},
        config=config
    )

    tokens = token_count(response['messages'])
    print(f'{city}: ~{tokens} tokens, {len(response['messages'])} messages')
    print(f'{(response['messages'])}')

Pulbangash: ~66 tokens, 4 messages
[HumanMessage(content='Find hotels in Pulbangash', additional_kwargs={}, response_metadata={}, id='3a3a297d-aab9-4190-b2d7-02a696a061ba'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-19T10:58:29.6141566Z', 'done': True, 'done_reason': 'stop', 'total_duration': 47002042600, 'load_duration': 14455707200, 'prompt_eval_count': 156, 'prompt_eval_duration': 23487002000, 'eval_count': 24, 'eval_duration': 9055115000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a153cf-62d2-7141-8e70-083d98464a5e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Pulbangash'}, 'id': '525ebf6d-1cab-4359-9138-ea5dec26dc88', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 24, 'total_tokens': 180}), ToolMessage(content='Hotels in Pulbangash', name='search_hotels', id='97033e2a-b699-4c7c-b03f-1f4b97b5305c', tool_c

#### Human in the loop middleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama

def read_mail(id:str) -> str:
    """Mock function to read emails by it's ID"""
    return f'Email content for ID: {id}'

def send_mail(recipient:str, subject:str, body:str) -> str:
    """Mock function to send email"""
    return f'Email sent to {recipient} with subject {subject}' 

agent = create_agent(
    model=model,
    tools=[read_mail,send_mail],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_mail":{"allowed_decisions":["approve", "edit", "reject"]},
            "read_mail":False
        }
    )]
)

In [ ]:
config = {
    "configurable":{"thread_id":"test_1"}
}

result = agent.invoke(
    {"message": [HumanMessage(content="Send email to srijan@gupta.com with subject 'hello")]},
    config=config
)